In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("./dataset/trump_tweets.csv")
df.columns

Index(['id', 'link', 'content', 'date', 'retweets', 'favorites', 'mentions',
       'hashtags', 'geo'],
      dtype='object')

In [3]:
df_text = df[['content']]
df_text.dropna()

,content
0,Be sure to tune in and watch Donald Trump on L...
1,Donald Trump will be appearing on The View tom...
2,Donald Trump reads Top Ten Financial Tips on L...
3,New Blog Post: Celebrity Apprentice Finale and...
4,"""My persona will never be that of a wallflower..."
...,...
41117,I have never seen the Republican Party as Stro...
41118,Now Mini Mike Bloomberg is critical of Jack Wi...
41119,I was thrilled to be back in the Great State o...
41120,"“In the House, the President got less due proc..."


## Remove unwanted text

In [4]:
import re
def remove_urls(text):
     """
     This function will try to remove URL present in out dataset and replace it with space using regex library.

     Input Args:
         text: strings of text that may contain URLs.

     Output Args:
         text: URLs replaces with text
     """
     url_pattern = re.compile(r'https?://\S+|www.\.\S+')
     return url_pattern.sub(r'', text)

### Check stuff

In [5]:
text = "Click here to open instagram https://instagram.com"
text_url = remove_urls(text)
text_url

'Click here to open instagram '

### apply and remove url 

In [6]:
text_no_url = df_text['content'].apply(remove_urls)

## Remove unwanted characters (emojis, flags, )

In [7]:
def remove_emoji(string):
  """
  This function will replace the emoji in string with whitespace
  """
  emoji_pattern = re.compile("["
                           u"\U0001F600-\U0001F64F"  # emoticons
                           u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                           u"\U0001F680-\U0001F6FF"  # transport & map symbols
                           u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                           u"\U00002702-\U000027B0"
                           u"\U000024C2-\U0001F251"
                           "]+", flags=re.UNICODE)
  return emoji_pattern.sub(r' ', string)

### check stuff

In [8]:
test_string = "Hello @swoyam 👋🏾, have you wondered about the meaning of life ever???  #Retard #Monday #🍱"
no_emoji = remove_emoji(test_string)
no_emoji

'Hello @swoyam  , have you wondered about the meaning of life ever???  #Retard #Monday # '

## remove every unwanted character  

In [9]:
def removeunwanted_characters(document):
    document = re.sub('@[A-Za-z0-9_]+', " ", document) #mentions
    document = re.sub('#[A-Za-z0-9_]+', " ", document) #hashtags 
    document = re.sub('[^A-Za-z0-9]+', " ", document)  #punctuations
    document = remove_emoji(document)     # emojis
    document = document.replace(" ", " ") # double spaces
    return document.strip()

### check stuff 

In [10]:
cleaned_string = removeunwanted_characters(test_string)
cleaned_string

'Hello have you wondered about the meaning of life ever'

### apply on data

In [11]:
text_removed_unwanted = df['content'].apply(removeunwanted_characters)
text_removed_unwanted

0        Be sure to tune in and watch Donald Trump on L...
1        Donald Trump will be appearing on The View tom...
2        Donald Trump reads Top Ten Financial Tips on L...
3        New Blog Post Celebrity Apprentice Finale and ...
4        My persona will never be that of a wallflower ...
                               ...                        
41117    I have never seen the Republican Party as Stro...
41118    Now Mini Mike Bloomberg is critical of Jack Wi...
41119    I was thrilled to be back in the Great State o...
41120    In the House the President got less due proces...
41121    A great show Check it out tonight at 9pm FoxNe...
Name: content, Length: 41122, dtype: object

## Tokenizations (using nltk)

In [12]:
!pip install nltk

In [13]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk import word_tokenize

[nltk_data] Downloading package punkt to /home/rokshh/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/rokshh/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


### check stuff

In [14]:
# Test case:
IN = "He did not try to navigate after the first bold flight, for the reaction had taken something out of his soul."
OUT = word_tokenize(IN)
OUT

['He',
 'did',
 'not',
 'try',
 'to',
 'navigate',
 'after',
 'the',
 'first',
 'bold',
 'flight',
 ',',
 'for',
 'the',
 'reaction',
 'had',
 'taken',
 'something',
 'out',
 'of',
 'his',
 'soul',
 '.']

## remove punctuations

In [15]:
from nltk.tokenize import RegexpTokenizer

def remove_punct(text):
    """
     This function removes the punctutations present in our text data.

     Input Args:
         text: text data.

     Returns:
         text: cleaned text.
    """
    tokenizer = RegexpTokenizer(r"\w+")
    lst = tokenizer.tokenize(" ".join(text))
    return lst

### check stuff

In [16]:
#Test
text_punctutation = "He did not try to navigate: after the!!!! first bold flight, for,,,,, the reaction!!!!had taken??????? something out of his soul."
text_punc_token = word_tokenize(text_punctutation)
print(text_punctutation)
print("-----------------------------------------------------------")
print(text_punc_token)
print("------------------------------------------------------------------")
text_clean = remove_punct(text_punc_token)
print(text_clean)

He did not try to navigate: after the!!!! first bold flight, for,,,,, the reaction!!!!had taken??????? something out of his soul.
-----------------------------------------------------------
['He', 'did', 'not', 'try', 'to', 'navigate', ':', 'after', 'the', '!', '!', '!', '!', 'first', 'bold', 'flight', ',', 'for', ',', ',', ',', ',', ',', 'the', 'reaction', '!', '!', '!', '!', 'had', 'taken', '?', '?', '?', '?', '?', '?', '?', 'something', 'out', 'of', 'his', 'soul', '.']
------------------------------------------------------------------
['He', 'did', 'not', 'try', 'to', 'navigate', 'after', 'the', 'first', 'bold', 'flight', 'for', 'the', 'reaction', 'had', 'taken', 'something', 'out', 'of', 'his', 'soul']


## Remove Stop word 

In [17]:
# download the stopwords list
nltk.download("stopwords")
from  nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# get the stopwords
stop_words = set(stopwords.words("english"))

# add custom stopwords
custom_stopwords = ['@', 'RT']
stop_words.update(custom_stopwords)

[nltk_data] Downloading package stopwords to /home/rokshh/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [18]:
def remove_stopwords(text_tokens):
    """
     This function removes all the stopwords present in out text tokens.
     
     Input Args:
         text_tokens: tokenize input of our datasets.
     Returns:
         result_tokens: list of token without stopword.
    """
    result_tokens = []
    for token in text_tokens:
        if token not in stop_words:
            result_tokens.append(token)
    return result_tokens

### Check the function

In [19]:
test_inputs = ['He', 'did', 'not', 'try', 'to', 'navigate', 'after', 'the', 'first', 'bold', 'flight', ',', 'for', 'the', 'reaction', 'had', 'taken', 'something', 'out', 'of', 'his', 'soul', '.']

print(test_inputs)
tokens_without_stopwords = remove_stopwords(test_inputs)
print(tokens_without_stopwords)

['He', 'did', 'not', 'try', 'to', 'navigate', 'after', 'the', 'first', 'bold', 'flight', ',', 'for', 'the', 'reaction', 'had', 'taken', 'something', 'out', 'of', 'his', 'soul', '.']
['He', 'try', 'navigate', 'first', 'bold', 'flight', ',', 'reaction', 'taken', 'something', 'soul', '.']


## Text Normalization

- reducing number of words present in corpus by process of lemmatization, stemming, Capital to Lower

### Lemmatization

- remove the forms keep only the root words 

In [20]:
from nltk.stem import WordNetLemmatizer
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')

def lemmatization(token_text):
    """
    This function performs the lemmatization operations as explained above.

    Input Args:
        token_text: list of tokens.

    Returns:
        lemmatized_tokens: list of lemmatized tokens.
    """
    wordnet = WordNetLemmatizer()
    lemmatized_tokens = [wordnet.lemmatize(token.lower(), pos = 'v') for token in token_text]
    return lemmatized_tokens

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/rokshh/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to /home/rokshh/nltk_data...


[nltk_data]   Package wordnet is already up-to-date!


### check the stuff

In [21]:
lemmatization("Should we go walking or swimming".split())

['should', 'we', 'go', 'walk', 'or', 'swim']

## Stemming
- token reduction technique that tries to chop off the tail part (doesn't take grammar into account)

In [22]:
from nltk.stem import PorterStemmer

def stemming(token_text):
    """
     This function performs stemming operations.

     Input Args:
         token_text: list of tokenize text.

     Returns:
         stemm_tokes: list of stemmed tokens.
     """
    porter = PorterStemmer()
    stemm_tokens = [porter.stem(token) for token in token_text]
    return stemm_tokens


In [23]:
#Test
print("-------------------------------" "INPUT TOKENS" "--------------------------------------------------------")
token_text_test=['Connects','Connecting','Connections','Connected','Connection','Connectings','Connect']
print(token_text_test)
print("------------------" "LEMMATIZED TOKENS" "-----------------------------------------------------------------")
lemma_tokens = lemmatization(token_text_test)
print(lemma_tokens)
print("---------------------" "STEMMED TOKENS" "-------------------------------------")
stemmed_tokens = stemming(token_text_test)
print(stemmed_tokens)

-------------------------------INPUT TOKENS--------------------------------------------------------
['Connects', 'Connecting', 'Connections', 'Connected', 'Connection', 'Connectings', 'Connect']
------------------LEMMATIZED TOKENS-----------------------------------------------------------------
['connect', 'connect', 'connections', 'connect', 'connection', 'connectings', 'connect']
---------------------STEMMED TOKENS-------------------------------------
['connect', 'connect', 'connect', 'connect', 'connect', 'connect', 'connect']


## Lower order 

In [24]:
def lower_order(text):
    """
    This function converts all the text in input text to lower order.

    Input Args:
        token_text : input text.

    Returns:
        small_order_text : text converted to small/lower order.
    """
    return  text.lower()

# Test:
sample_text = "This Is some Normalized TEXT"
sample_small = lower_order(sample_text)
print(sample_small)

this is some normalized text


# Input text pipeline

In [25]:
data = pd.read_csv("./dataset/trum_tweet_sentiment_analysis.csv")
data.head

<bound method NDFrame.head of                                                       text  Sentiment
0        RT @JohnLeguizamo: #trump not draining swamp b...          0
1        ICYMI: Hackers Rig FM Radio Stations To Play A...          0
2        Trump protests: LGBTQ rally in New York https:...          1
3        "Hi I'm Piers Morgan. David Beckham is awful b...          0
4        RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...          0
...                                                    ...        ...
1850118  Everytime im like 'How the fuck I follow Melan...          0
1850119  RT @imgur: The Trump Handshake. https://t.co/R...          0
1850120  "Greenspan warns Trump's policies risk inflati...          0
1850121  RT @FasinatingLogic: We must also #INVESTIGATE...          1
1850122  RT @imgur: The Trump Handshake. https://t.co/R...          0

[1850123 rows x 2 columns]>

### data Cleaning

In [26]:
data_cleaning = data["text"].dropna()
data_cleaning[0]

'RT @JohnLeguizamo: #trump not draining swamp but our taxpayer dollars on his trips to advertise his properties! @realDonaldTrump\x85 https://t.co/gFBvUkMX9z'

### text Cleaning pipeline

In [27]:
def text_cleaning_pipeline(dataset, rule = "lemmatize"):
  """
  This function cleans up the text, changes order, remove url, emojis and unwanted characters
  turns them to token and then lemmatize or stemm them according to the rule passed

  Input Args:
    dataset: the data col to perform these operations on
    rule: Whether to lemmatize or stemm it (Default is lemmatize)

  Output:
    cleaned list of tokens
  """
  # Convert the input to small/lower order.
  data = lower_order(dataset)
  # Remove URLs
  data = remove_urls(dataset)
  # Remove emojis
  data = remove_emoji(dataset)
  # Remove all other unwanted characters.
  data = removeunwanted_characters(dataset)
  # Create tokens.
  tokens = data.split()

  if rule == "lemmatize":
    tokens = lemmatization(tokens)
  elif rule == "stem":
    tokens = stemming(tokens)
  else:
    print("Pick between lemmatize or stem")

  return " ".join(tokens)


### check pipeline

In [28]:
sample = "Hello @gabe_flomo 👋🏾, I still want us to hit that new sushi spot??? LMK when you're free cuz I can't go this or next weekend since I'll be swimming!!! #sushiBros #rawFish #🍱"

text_cleaning_pipeline(sample)

'hello i still want us to hit that new sushi spot lmk when you re free cuz i can t go this or next weekend since i ll be swim'

###  apply on real data 

In [29]:
test = data["text"][0]

In [30]:
print(text_cleaning_pipeline(test))

rt not drain swamp but our taxpayer dollars on his trip to advertise his properties https t co gfbvukmx9z


In [31]:
cleaned_tokens = data["text"].apply(lambda dataset: text_cleaning_pipeline(dataset))

### Check if the tokens are valid

In [32]:
cleaned_tokens[0]

'rt not drain swamp but our taxpayer dollars on his trip to advertise his properties https t co gfbvukmx9z'

# Text Classification using ML models

In [40]:
print(data.columns)
print(type(cleaned_tokens[0]))  # should be str, not list

Index(['text', 'Sentiment'], dtype='object')
<class 'str'>


## Train test split

In [41]:
X = cleaned_tokens          # processed text (features)
y = data["Sentiment"]           # labels (targets)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
) 

print("X Train shape", X_train.shape)
print("X Test shape", X_test.shape)

print("Y Train shape", y_train.shape)
print("Y Test shape", y_test.shape)

X Train shape (1480098,)
X Test shape (370025,)
Y Train shape (1480098,)
Y Test shape (370025,)


## Tfidf for data

In [42]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
   max_features=10000,   # cap vocabulary size
   min_df=5,             # ignore very rare terms
   max_df=0.95,          # ignore near-universal terms
   stop_words='english'
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)         # transform ONLY, never fit on test

# View the feature names (vocabulary)
print("Feature Names:", vectorizer.get_feature_names_out())

# View the shape of the resulting sparse matrix
print("Matrix Shape:", X_train_tfidf.shape)

# View the TF-IDF values as a dense array
# print("TF-IDF Matrix:\n", X.toarray())

Feature Names: ['00' '000' '00aaickbos' ... 'zxbbcu7sgh' 'zxs4xdhb7p' 'zyaeviw45g']
Matrix Shape: (1480098, 10000)


## import model and train

In [44]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# train
model = LogisticRegression(max_iter=1000, random_state=42, solver='saga', n_jobs=-1)
model.fit(X_train_tfidf, y_train)

# predict
y_pred = model.predict(X_test_tfidf)

# evaluate
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.94      0.96      0.95    248563
           1       0.91      0.87      0.89    121462

    accuracy                           0.93    370025
   macro avg       0.92      0.92      0.92    370025
weighted avg       0.93      0.93      0.93    370025

